# 08 - Model Deployment
**Phase 2: Deploy model for batch inference on Azure**

### Deployment Strategy: Batch Inference
- Generate recommendations for ALL customers periodically
- Save results to Azure Data Lake for downstream consumption
- Suitable for: email campaigns, app home pages, weekly updates

### Why Batch (not Real-time)?
- 1.3M customers × 100K articles = too large for real-time
- Recommendations don't need to update instantly
- Cost-effective and scalable

In [0]:
# ========================================
# SETUP & CONFIGURATION
# ========================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, date
import mlflow

# Storage Configuration
STORAGE_ACCOUNT = "ragadziyada"
STORAGE_KEY = "qIXjwo7EGK8an84BPCk446tqY9L7K7r3a9W2WB+voe2vUCvW1SK3qc/7UGOicKyBAtrptYcVfTB7+AStvWZi0A=="

CURATED_CONTAINER = "curated-p1"

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

# Paths
CUSTOMER_FEATURES = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/customer_features/"
MODEL_ARTIFACT = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/models/personalized_recommender_v1/"
PREDICTIONS_OUTPUT = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/predictions/"

K = 12

print("✅ Configuration loaded!")
print(f"   Model artifact: {MODEL_ARTIFACT}")
print(f"   Output path: {PREDICTIONS_OUTPUT}")

✅ Configuration loaded!
   Model artifact: abfss://curated-p1@ragadziyada.dfs.core.windows.net/models/personalized_recommender_v1/
   Output path: abfss://curated-p1@ragadziyada.dfs.core.windows.net/predictions/


In [0]:
# ========================================
# LOAD MODEL & CUSTOMER DATA
# ========================================

# Load the model artifact (department recommendations lookup table)
dept_recommendations_df = spark.read.parquet(MODEL_ARTIFACT)

print(f"✅ Model artifact loaded!")
print(f"   Departments: {dept_recommendations_df.select('article_department_name').distinct().count()}")
print(f"   Total article recommendations: {dept_recommendations_df.count()}")

# Load customer features
customer_features_df = spark.read.parquet(CUSTOMER_FEATURES)

print(f"\n✅ Customer features loaded!")
print(f"   Total customers: {customer_features_df.count():,}")

# Preview model artifact
print(f"\n📋 Model artifact sample:")
display(dept_recommendations_df.orderBy("article_department_name", "dept_rank").limit(10))

✅ Model artifact loaded!
   Departments: 250
   Total article recommendations: 2862

✅ Customer features loaded!
   Total customers: 1,362,281

📋 Model artifact sample:


article_department_name,article_id,dept_sales,dept_rank
AK Bottoms,0664871001,891,1
AK Bottoms,0717797001,487,2
AK Bottoms,0858692004,392,3
AK Bottoms,0708679002,376,4
AK Bottoms,0863502004,265,5
AK Bottoms,0863502005,237,6
AK Bottoms,0834069002,226,7
AK Bottoms,0858692003,218,8
AK Bottoms,0735740001,215,9
AK Bottoms,0749147002,213,10


In [0]:
# ========================================
# GENERATE BATCH PREDICTIONS
# ========================================
# Generate top-12 recommendations for ALL customers

print("🚀 Generating recommendations for all customers...")
start_time = datetime.now()

# Prepare department recommendations as lookup
dept_recs_array = (
    dept_recommendations_df
    .groupBy("article_department_name")
    .agg(F.collect_list("article_id").alias("recommended_articles"))
)

# Get global top-12 for customers without a preferred department
global_top_12 = (
    dept_recommendations_df
    .groupBy("article_id")
    .agg(F.sum("dept_sales").alias("total_sales"))
    .orderBy(F.desc("total_sales"))
    .limit(K)
)
global_top_12_list = [row.article_id for row in global_top_12.collect()]

# Join customers with their department recommendations
batch_predictions = (
    customer_features_df
    .select("customer_id", "customer_preferred_department")
    .join(
        dept_recs_array,
        customer_features_df.customer_preferred_department == dept_recs_array.article_department_name,
        how="left"
    )
    .select(
        "customer_id",
        "customer_preferred_department",
        "recommended_articles"
    )
)

# Fill nulls with global top-12
batch_predictions = batch_predictions.withColumn(
    "recommended_articles",
    F.when(
        F.col("recommended_articles").isNull(),
        F.array([F.lit(x) for x in global_top_12_list])
    ).otherwise(F.col("recommended_articles"))
)

# Add metadata
batch_predictions = batch_predictions.withColumn(
    "prediction_date", F.lit(datetime.now().date().isoformat())
)

prediction_count = batch_predictions.count()
elapsed = (datetime.now() - start_time).total_seconds()

print(f"✅ Batch predictions generated!")
print(f"   Customers: {prediction_count:,}")
print(f"   Time: {elapsed:.1f} seconds")

display(batch_predictions.limit(5))

🚀 Generating recommendations for all customers...
✅ Batch predictions generated!
   Customers: 1,362,281
   Time: 1.6 seconds


customer_id,customer_preferred_department,recommended_articles,prediction_date
0003e9bbb9faf3937ad3a28a5bede5c1b896c1bc6c10354ed3487a5ffb4dd30e,Swimwear,"List(0351484002, 0688537004, 0590928001, 0599580017, 0684209004, 0688537011, 0600886001, 0684209013, 0699080001, 0599580038, 0723529001, 0689109001)",2026-04-10
000749135ee9aa3a24c2316ea5ae4f495b39c1653c5612bb5b239f1b2a182a2a,Jersey Basic,"List(0610776002, 0610776001, 0565379001, 0554598001, 0179123001, 0803757001, 0778064003, 0778064001, 0787946002, 0841383002, 0565379002, 0768912001)",2026-04-10
000c886e014a122bd9066501103e3f4a3ec157af27399a5f6fa2dc540c123356,Swimwear,"List(0351484002, 0688537004, 0590928001, 0599580017, 0684209004, 0688537011, 0600886001, 0684209013, 0699080001, 0599580038, 0723529001, 0689109001)",2026-04-10
00108c4b7847b71af7540e34b9842b1a18b5c5ee28e9481c07ccbe2dda375363,Blouse,"List(0507909001, 0695632002, 0695632001, 0507910001, 0762846001, 0716672001, 0762846008, 0618800001, 0850917001, 0762846006, 0507909003, 0697054014)",2026-04-10
0022efaae249604b1f068d1b64d1e259beec1fbaf0f1f5d4c285f322dadb792c,Blouse,"List(0507909001, 0695632002, 0695632001, 0507910001, 0762846001, 0716672001, 0762846008, 0618800001, 0850917001, 0762846006, 0507909003, 0697054014)",2026-04-10


In [0]:
# ========================================
# DEPLOYMENT INPUT/OUTPUT SPECIFICATION
# ========================================

print("""
┌─────────────────────────────────────────────────────────────────────────┐
│                    DEPLOYMENT SPECIFICATION                             │
└─────────────────────────────────────────────────────────────────────────┘

📥 INPUT FORMAT (Customer Features):
───────────────────────────────────────────────────────────────────────────
{
    "customer_id": "0003e9bbb9faf3937ad3a28a5bede5c1b896c1bc6c10354ed3487a5ffb4dd30e",
    "customer_preferred_department": "Swimwear"
}

📤 OUTPUT FORMAT (Recommendations):
───────────────────────────────────────────────────────────────────────────
{
    "customer_id": "0003e9bbb9faf3937ad3a28a5bede5c1b896c1bc6c10354ed3487a5ffb4dd30e",
    "customer_preferred_department": "Swimwear",
    "recommended_articles": [
        "0351484002", "0688537004", "0590928001", "0599580017",
        "0684209004", "0688537011", "0600886001", "0684209013",
        "0699080001", "0599580038", "0723529001", "0689109001"
    ],
    "prediction_date": "2026-04-10"
}

📋 FIELD DEFINITIONS:
───────────────────────────────────────────────────────────────────────────
- customer_id (string): SHA-256 hashed customer identifier
- customer_preferred_department (string): Most frequently purchased department
- recommended_articles (array[string]): Top-12 article IDs to recommend
- prediction_date (string): Date predictions were generated (YYYY-MM-DD)

🔄 INFERENCE LOGIC:
───────────────────────────────────────────────────────────────────────────
1. Lookup customer's preferred_department
2. Retrieve top-12 popular articles from that department
3. If department not found → fallback to global top-12
4. Return recommendation array
""")

# Show actual example
print("\n📋 ACTUAL EXAMPLE FROM BATCH:")
example = batch_predictions.limit(1).collect()[0]
print(f"""
Input:
  customer_id: {example['customer_id'][:20]}...
  preferred_department: {example['customer_preferred_department']}

Output:
  recommended_articles: {example['recommended_articles'][:4]}... (12 total)
  prediction_date: {example['prediction_date']}
""")


┌─────────────────────────────────────────────────────────────────────────┐
│                    DEPLOYMENT SPECIFICATION                             │
└─────────────────────────────────────────────────────────────────────────┘

📥 INPUT FORMAT (Customer Features):
───────────────────────────────────────────────────────────────────────────
{
    "customer_id": "0003e9bbb9faf3937ad3a28a5bede5c1b896c1bc6c10354ed3487a5ffb4dd30e",
    "customer_preferred_department": "Swimwear"
}

📤 OUTPUT FORMAT (Recommendations):
───────────────────────────────────────────────────────────────────────────
{
    "customer_id": "0003e9bbb9faf3937ad3a28a5bede5c1b896c1bc6c10354ed3487a5ffb4dd30e",
    "customer_preferred_department": "Swimwear",
    "recommended_articles": [
        "0351484002", "0688537004", "0590928001", "0599580017",
        "0684209004", "0688537011", "0600886001", "0684209013",
        "0699080001", "0599580038", "0723529001", "0689109001"
    ],
    "prediction_date": "2026-04-10"
}

📋 

In [0]:
# ========================================
# SAVE PREDICTIONS TO AZURE
# ========================================

# Save with date partitioning for versioning
prediction_date = datetime.now().strftime("%Y-%m-%d")
output_path = f"{PREDICTIONS_OUTPUT}batch_{prediction_date}/"

batch_predictions.write.mode("overwrite").parquet(output_path)

print(f"✅ Predictions saved to Azure Data Lake!")
print(f"   Path: {output_path}")
print(f"   Customers: {prediction_count:,}")
print(f"   Date: {prediction_date}")

# Also save a "latest" version for easy access
latest_path = f"{PREDICTIONS_OUTPUT}latest/"
batch_predictions.write.mode("overwrite").parquet(latest_path)

print(f"\n✅ Latest predictions also saved!")
print(f"   Path: {latest_path}")

✅ Predictions saved to Azure Data Lake!
   Path: abfss://curated-p1@ragadziyada.dfs.core.windows.net/predictions/batch_2026-04-10/
   Customers: 1,362,281
   Date: 2026-04-10

✅ Latest predictions also saved!
   Path: abfss://curated-p1@ragadziyada.dfs.core.windows.net/predictions/latest/


In [0]:
# ========================================
# CREATE DEPLOYMENT CONFIGURATION
# ========================================

deployment_config = {
    "deployment_type": "batch_inference",
    "model_name": "hm_personalized_recommender_60304248",
    "model_artifact_path": MODEL_ARTIFACT,
    "predictions_output_path": PREDICTIONS_OUTPUT,
    
    "schedule": {
        "frequency": "weekly",
        "description": "Generate fresh recommendations every week"
    },
    
    "input": {
        "customer_features": CUSTOMER_FEATURES,
        "total_customers": prediction_count
    },
    
    "output": {
        "format": "parquet",
        "partitioning": "by_date",
        "latest_path": f"{PREDICTIONS_OUTPUT}latest/"
    },
    
    "deployed_by": "Team 60304248",
    "deployed_date": datetime.now().isoformat()
}

print("📋 DEPLOYMENT CONFIGURATION")
print("=" * 50)
for key, value in deployment_config.items():
    if isinstance(value, dict):
        print(f"\n{key}:")
        for k, v in value.items():
            print(f"   {k}: {v}")
    else:
        print(f"{key}: {value}")

print("\n✅ Configuration ready for Azure DevOps pipeline!")

📋 DEPLOYMENT CONFIGURATION
deployment_type: batch_inference
model_name: hm_personalized_recommender_60304248
model_artifact_path: abfss://curated-p1@ragadziyada.dfs.core.windows.net/models/personalized_recommender_v1/
predictions_output_path: abfss://curated-p1@ragadziyada.dfs.core.windows.net/predictions/

schedule:
   frequency: weekly
   description: Generate fresh recommendations every week

input:
   customer_features: abfss://curated-p1@ragadziyada.dfs.core.windows.net/customer_features/
   total_customers: 1362281

output:
   format: parquet
   partitioning: by_date
   latest_path: abfss://curated-p1@ragadziyada.dfs.core.windows.net/predictions/latest/
deployed_by: Team 60304248
deployed_date: 2026-04-10T10:45:03.699911

✅ Configuration ready for Azure DevOps pipeline!


# 📊 Notebook 08 Summary

## Deployment Complete!

| Item | Value |
|------|-------|
| Deployment Type | Batch Inference |
| Customers Served | 1,362,281 |
| Recommendations per Customer | 12 |
| Prediction Time | ~2 seconds |

## Output Locations

| Path | Purpose |
|------|---------|
| `predictions/batch_2026-04-10/` | Today's batch predictions |
| `predictions/latest/` | Always points to most recent predictions |

## How It Works
1. Load model artifact (department recommendations lookup)
2. Load all customer features
3. Match each customer to their preferred department
4. Return top-12 items from that department
5. Save to Azure Data Lake

## Deployment Schedule
- **Frequency:** Weekly
- **Trigger:** Can be automated via Databricks Jobs or Azure DevOps Pipeline

## Next Steps
- **Notebook 09:** Validate deployed predictions
- **Azure DevOps:** Create CI/CD pipeline for automation

---
*Phase 2 - Deployment | H&M Fashion Recommendations | Team 60304248*